# Analyse Big Data des accidents de la route avec PySpark

*HAMRIT Wassim*

## Introduction
Ce notebook présente une analyse des données relatives aux accidents de la route en Australie-Méridionale, fournies par le Department of Planning, Transport and Infrastructure (DPTI). L'objectif est d'utiliser PySpark pour effectuer des requêtes analytiques sur ces données massives (2012-2019) via deux API distinctes : l'API RDD (bas niveau) et l'API DataFrame (haut niveau).

In [1]:
import sys
import os
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, FloatType, StringType
from pyspark.sql.functions import udf, col
import pandas as pd
import matplotlib.pyplot as plt

# Initialisation de Spark
conf = SparkConf().setAppName("RoadCrashAnalysis").setMaster("local[*]")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

## 1. Analyse des données avec les RDDs
Dans cette section, nous créons des RDDs (Resilient Distributed Datasets) à partir des fichiers CSV, gérons le partitionnement et effectuons des requêtes bas niveau.

In [2]:
# Chargement des fichiers CSV dans des RDDs
units_rdd = sc.textFile('./units_data/*.csv')
crash_rdd = sc.textFile('./crash_data/*.csv')

# Suppression des en-têtes
units_header = units_rdd.first()
units_rdd = units_rdd.filter(lambda row: row != units_header)
print("Total count of units_rdd: ", units_rdd.count())

crash_header = crash_rdd.first()
crash_rdd = crash_rdd.filter(lambda row: row != crash_header)
print("Total count of crash_rdd: ", crash_rdd.count())

Total count of units_rdd:  272591
Total count of crash_rdd:  127672


### Partitionnement personnalisé
Nous partitionnons les données en deux : une partition pour les véhicules d'Australie-Méridionale (SA) et une autre pour les autres états.

In [3]:
def parseRecord(line):
    lines = line.split(',')
    array_line = [str(value).replace('"', '') for value in lines]
    return (array_line[3], [x for i, x in enumerate(array_line) if i != 3])

units_state_rdd = units_rdd.map(parseRecord)

def hash_SA(key):
    return 0 if key == 'SA' else 1
    
SA_partitioned_rdd = units_state_rdd.partitionBy(2, hash_SA)

def print_partitions(data):
    numPartitions = data.getNumPartitions()
    partitions = data.glom().collect()
    print(f"NUMBER OF PARTITIONS: {numPartitions}")
    for index, partition in enumerate(partitions):
        if len(partition) > 0:
            print(f"Partition {index}: {len(partition)} records")

print_partitions(SA_partitioned_rdd)

NUMBER OF PARTITIONS: 2
Partition 0: 218366 records
Partition 1: 54225 records


### Requêtes RDD
**1. Âge moyen des conducteurs selon leur genre.**

In [4]:
def parseGenderAge(line):
    lines = line.split(',')
    array_line = [str(value).replace('"', '') for value in lines]
    if (array_line[8] != 'XXX') & (array_line[8] != '') & (array_line[7] != 'Unknown'):
        return (array_line[7], int(array_line[8]))

gender_age_rdd = units_rdd.map(parseGenderAge)
filtered_gender_age_rdd = gender_age_rdd.filter(lambda x: x is not None).filter(lambda x: x != "")
gender_age_avg_rdd = filtered_gender_age_rdd.groupByKey().mapValues(lambda x: round(sum(x) / len(x), 2))
gender_age_avg_rdd.collect()

[('Male', 40.8), ('Female', 40.08)]

**2. Véhicules les plus anciens et les plus récents impliqués dans un accident.**

In [5]:
def parseYear(line):
    lines = line.split(',')
    array_line = [str(value).replace('"', '') for value in lines]
    if (array_line[5] != 'XXXX') & (array_line[5] != ''):
        return int(array_line[5]), [array_line[3], array_line[4]]

year_rdd = units_rdd.map(parseYear)
clean_year_rdd = year_rdd.filter(lambda x: x is not None).filter(lambda x: x != "")

print("Oldest year 5 crashes:\n", clean_year_rdd.sortByKey(ascending=True).collect()[0:5])
print("\n")
print("Newest year 5 crashes:\n", clean_year_rdd.sortByKey(ascending=False).collect()[0:5])

Oldest year 5 crashes:
 [(1900, ['VIC', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle'])]


Newest year 5 crashes:
 [(2019, ['SA', 'Station Wagon']), (2019, ['SA', 'OMNIBUS']), (2019, ['SA', 'Motor Cars - Sedan']), (2019, ['SA', 'Station Wagon']), (2019, ['SA', 'SEMI TRAILER'])]


---
## 2. Analyse des données avec les DataFrames PySpark


In [6]:
# Chargement avec l'API DataFrame
units_df = spark.read.format('csv')\
            .option('header',True).option('escape','"')\
            .load('./units_data/*.csv')

crash_df = spark.read.format('csv')\
            .option('header',True).option('escape','"')\
            .load('./crash_data/*.csv')

### Requêtes d'analyse avec les DataFrames
**1. Trouver tous les accidents à Adélaïde causant plus de 3 victimes.**

In [7]:
crash_df = crash_df.withColumn('Total Cas', F.col('Total Cas').cast(IntegerType()))
adelaide_crash_casualty_df = crash_df.filter(col("Total Cas") > 3).filter(col("Suburb") == 'ADELAIDE')
adelaide_crash_casualty_df.collect()

[Row(REPORT_ID='2012-4842-21/08/2019', Stats Area='1 City', Suburb='ADELAIDE', Postcode='5000', LGA Name='CITY OF ADELAIDE', Total Units='2', Total Cas=4, Total Fats='0', Total SI='0', Total MI='4', Year='2012', Month='March', Day='Thursday', Time='04:00 pm', Area Speed='050', Position Type='Not Divided', Horizontal Align='Straight road', Vertical Align='Level', Other Feat='Not Applicable', Road Surface='Sealed', Moisture Cond='Dry', Weather Cond='Not Raining', DayNight='Daylight', Crash Type='Rear End', Unit Resp='01', Entity Code='Driver Rider', CSEF Severity='2: MI', Traffic Ctrls='No Control', DUI Involved=None, Drugs Involved=None, ACCLOC_X='1327421.54', ACCLOC_Y='1669848.73', UNIQUE_LOC='13274221669849'),
 Row(REPORT_ID='2012-5769-21/08/2019', Stats Area='1 City', Suburb='ADELAIDE', Postcode='5000', LGA Name='CITY OF ADELAIDE', Total Units='2', Total Cas=4, Total Fats='0', Total SI='0', Total MI='4', Year='2012', Month='March', Day='Friday', Time='09:30 am', Area Speed='050', Pos

**2. Affichage des 10 accidents causant le plus de victimes.**

In [8]:
crash_df.sort(col('Total Cas'), ascending=False).show(10)

+--------------------+--------------+-------------+--------+--------------------+-----------+---------+----------+--------+--------+----+--------+---------+--------+----------+-------------+--------------------+--------------+--------------+------------+-------------+------------+--------+-----------+---------+------------+-------------+---------------+------------+--------------+----------+----------+--------------+
|           REPORT_ID|    Stats Area|       Suburb|Postcode|            LGA Name|Total Units|Total Cas|Total Fats|Total SI|Total MI|Year|   Month|      Day|    Time|Area Speed|Position Type|    Horizontal Align|Vertical Align|    Other Feat|Road Surface|Moisture Cond|Weather Cond|DayNight| Crash Type|Unit Resp| Entity Code|CSEF Severity|  Traffic Ctrls|DUI Involved|Drugs Involved|  ACCLOC_X|  ACCLOC_Y|    UNIQUE_LOC|
+--------------------+--------------+-------------+--------+--------------------+-----------+---------+----------+--------+--------+----+--------+---------+--

**3. Décompte du nombre total de décès par type d'accident.**

In [9]:
crash_df.groupBy('Crash Type').agg(F.sum('Total Fats').alias('Total fatalities')).sort(col('Total fatalities'), ascending=False).show()

+--------------------+----------------+
|          Crash Type|Total fatalities|
+--------------------+----------------+
|    Hit Fixed Object|           245.0|
|             Head On|           136.0|
|      Hit Pedestrian|           109.0|
|           Roll Over|            93.0|
|         Right Angle|            82.0|
|            Rear End|            33.0|
|          Side Swipe|            30.0|
|          Right Turn|            26.0|
|  Hit Parked Vehicle|            10.0|
|          Hit Animal|             7.0|
|               Other|             5.0|
|Left Road - Out o...|             3.0|
|  Hit Object on Road|             2.0|
+--------------------+----------------+



**4. Affichage des banlieues avec le plus grand nombre de victimes causées par des conducteurs sans permis.**

In [10]:
complete_df = crash_df.join(units_df, crash_df.REPORT_ID == units_df.REPORT_ID, how='inner')
by_suburb_no_lic_no_df = complete_df.filter(col('Licence Type') == 'Unlicenced').groupby('Suburb').agg(F.sum('Total Cas').alias('Total casualty'))
by_suburb_no_lic_no_df.sort(col('Total casualty'), ascending=False).show()

+---------------+--------------+
|         Suburb|Total casualty|
+---------------+--------------+
|       ADELAIDE|            43|
|      SALISBURY|            30|
|       PROSPECT|            27|
|     INGLE FARM|            22|
| SALISBURY EAST|            20|
|  MORPHETT VALE|            20|
|      DRY CREEK|            19|
|  MURRAY BRIDGE|            19|
| NORTH ADELAIDE|            17|
|   MAWSON LAKES|            17|
|        ENFIELD|            16|
|SALISBURY DOWNS|            15|
|    GEPPS CROSS|            15|
|   REGENCY PARK|            15|
| ELIZABETH PARK|            14|
|PARA HILLS WEST|            14|
|         SEATON|            14|
|   BEDFORD PARK|            13|
|   DAVOREN PARK|            13|
|  MOUNT GAMBIER|            13|
+---------------+--------------+
only showing top 20 rows



### Impact de l'alcool et des drogues sur la sévérité des accidents

In [11]:
drugs_positive_df = crash_df.filter(col('Drugs Involved') == 'Y').groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Drugs'))
alcohol_positive_df = crash_df.filter(col('DUI Involved') == 'Y').groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Alcohol'))
alcohol_drugs_positive_df = crash_df.filter((col('DUI Involved') == 'Y') & (col('Drugs Involved') == 'Y')).groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Both'))
without_drugs_alcohol_df = crash_df.filter((col('DUI Involved').isNull()) & (col('Drugs Involved').isNull()))
drugs_alcohol_negative_df = without_drugs_alcohol_df.groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On None'))

all_df = drugs_positive_df.join(alcohol_positive_df, ["CSEF Severity"]).join(alcohol_drugs_positive_df, ["CSEF Severity"]).join(drugs_alcohol_negative_df, ["CSEF Severity"])

total_on_drugs = drugs_positive_df.groupBy().sum().collect()[0][0]
total_on_alcohol = alcohol_positive_df.groupBy().sum().collect()[0][0]
total_on_both = alcohol_drugs_positive_df.groupBy().sum().collect()[0][0]
total_on_none = drugs_alcohol_negative_df.groupBy().sum().collect()[0][0]

def to_percent(val, total): return float(round((int(val)/total)*100,2))

all_df_per = all_df.withColumn('On Drugs', udf(lambda x: to_percent(x, total_on_drugs), FloatType())('On Drugs')) \
                   .withColumn('On Alcohol', udf(lambda x: to_percent(x, total_on_alcohol), FloatType())('On Alcohol')) \
                   .withColumn('On Both', udf(lambda x: to_percent(x, total_on_both), FloatType())('On Both')) \
                   .withColumn('On None', udf(lambda x: to_percent(x, total_on_none), FloatType())('On None'))

all_df_per.show()

+-------------+--------+----------+-------+-------+
|CSEF Severity|On Drugs|On Alcohol|On Both|On None|
+-------------+--------+----------+-------+-------+
|     4: Fatal|    6.31|      3.42|  13.09|   0.42|
|        2: MI|   59.82|     31.24|  50.67|  28.74|
|       1: PDO|   12.78|     53.88|  12.08|  67.49|
|        3: SI|   21.09|     11.46|  24.16|   3.36|
+-------------+--------+----------+-------+-------+



*Commentaire : On observe clairement que les accidents fatals (4: Fatal) représentent une part beaucoup plus importante (13.09%) lorsque le conducteur cumule drogue et alcool, contre seulement 0.42% en l'absence de substance. A l'inverse, sans substance, une large majorité (67.49%) des accidents n'entraînent que des dégâts matériels (1: PDO - Property Damage Only).*

### Visualisation des résultats

In [12]:
pd_df_per = all_df_per.toPandas()
pd_df_per.set_index('CSEF Severity').plot(kind='bar', figsize=(10, 6), colormap='viridis')
plt.title("Impact de la drogue et de l'alcool sur la sévérité des accidents")
plt.ylabel("Pourcentage (%)")
plt.xticks(rotation=0)
plt.legend(title='Etat du conducteur')
plt.tight_layout()
plt.show()